# EPL dataset audit and training handoff

This companion rebuilds and inspects the saved dataset. The scripts were executed and verified when this release was created; run these cells in your own environment. The final test set is not scored. See README.md for assumptions and source permissions.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'source_manifest.json').exists():
    ROOT = ROOT / 'arsenal_predictor_v1'
assert (ROOT / 'source_manifest.json').exists(), 'Set ROOT to the extracted folder'


## 1. Rebuild from the frozen inputs
No network access is required while the bundled raw snapshots are present.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/build_dataset.py')], check=True)


## 2. Inspect coverage and reconciliation
The 3,800 overlapping matches are not appended as new training examples.

In [ ]:
summary = json.loads((ROOT / 'audit/summary.json').read_text())
display(pd.Series(summary))
display(pd.read_csv(ROOT / 'audit/season_coverage.csv'))
display(pd.read_csv(ROOT / 'audit/source_conflicts.csv'))
display(pd.read_csv(ROOT / 'audit/invalid_shot_measurements.csv'))


## 3. Inspect permitted predictors
Only the allowlisted numeric columns enter the default predictor. Same-match statistics are targets or historical inputs for later matches.

In [ ]:
schema = json.loads((ROOT / 'feature_schema.json').read_text())
X = pd.read_csv(ROOT / 'data/train/X.csv')
y = pd.read_csv(ROOT / 'data/train/y.csv')['target_result']
assert X.columns.tolist() == schema['features']
print(X.shape, y.value_counts().to_dict())
display(pd.read_csv(ROOT / 'data_dictionary.csv'))


## 4. Verify temporal isolation and model compatibility
This fits a fixed logistic model on train and checks validation. It does not score the reserved test partition.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/verify_dataset.py')], check=True)
display(json.loads((ROOT / 'audit/validation.json').read_text()))


## 5. Arsenal-specific view
This is a publication/research view, not an all-columns training matrix. Probabilities for home and away wins must be mapped to Arsenal according to venue.

In [ ]:
arsenal = pd.read_csv(ROOT / 'data/arsenal_view.csv')
display(arsenal.groupby('split').size())
display(arsenal[['date','home_team','away_team','arsenal_result','split']].head())
